In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :sliding
KMAX_UPPER = 30  
KMAX_fixed = 22

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,
            k_max_fixed = KMAX_fixed)
    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding_model] Fitting chain 3 (tau=59)
[ Info: [sliding] iter 1000/1000000 elapsed=5.7s, rate=0.179, mean=[0.772, 0.00224, 1.179], std=[0.0385, 0.000798, 0.1633] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=10.4s, rate=0.142, mean=[0.805, 0.00138, 1.168], std=[0.0409, 0.000963, 0.1162] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=14.4s, rate=0.124, mean=[0.815, 0.00107, 1.149], std=[0.0376, 0.000878, 0.0985] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=18.4s, rate=0.115, mean=[0.818, 0.00091, 1.124], std=[0.0331, 0.000804, 0.0945] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=22.4s, rate=0.107, mean=[0.824, 0.00081, 1.114], std=[0.0317, 0.000742, 0.0863] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=26.4s, rate=0.101, mean=[0.830, 0.00074, 1.113], std=[0.0314, 0.000692, 0.0790] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=30.3s, rate=0.095, mean=[0.838, 0.00069, 1.113], std=[0.0342, 0.000652, 0.0732] [ADAPT]
[ Info: [sliding] iter 8000/